# 14.4 Retraining Pipelines

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 14: MLOps & Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-14-mlops-and-monitoring/14.04_retraining-pipelines.ipynb)

Companion code for Section 14.4 of the book. Run the cells in order from top to bottom.

**Example 14.4.1** · bash

```bash
import numpy as np
import pandas as pd
import joblib
from datetime import datetime, timedelta
from typing import Optional, Dict, Tuple
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

# ── Champion-Challenger Framework ─────────────────────────────────
class ChampionChallengerFramework:
    '''
    Manages champion model promotion.
    Champion = currently deployed model.
    Challenger = newly trained candidate.
    '''

    def __init__(self,
                  min_auc_improvement: float = 0.005,
                  min_test_samples:    int   = 100,
                  shadow_mode:         bool  = True):
        self.min_improvement = min_auc_improvement
        self.min_samples     = min_test_samples
        self.shadow_mode     = shadow_mode
        self.champion        = None
        self.champion_auc    = None
        self.history         = []

    def set_champion(self, model, auc: float, version: str) -> None:
        self.champion     = model
        self.champion_auc = auc
        self.champion_ver = version
        print(f'Champion set: {version}  AUC={auc:.4f}')

    def evaluate_challenger(self,
                              challenger,
                              X_holdout: np.ndarray,
                              y_holdout: np.ndarray,
                              challenger_version: str) -> Dict:
        if len(y_holdout) < self.min_samples:
            return {'decision': 'SKIP', 'reason': f'Only {len(y_holdout)} samples < {self.min_samples}'  }

        challenger_auc = roc_auc_score(y_holdout, challenger.predict_proba(X_holdout)[:,1])
        champion_auc   = roc_auc_score(y_holdout, self.champion.predict_proba(X_holdout)[:,1])
        improvement    = challenger_auc - champion_auc

        should_promote = improvement >= self.min_improvement

        result = {
            'timestamp':         datetime.utcnow().isoformat(),
            'challenger_version':challenger_version,
            'champion_version':  self.champion_ver,
            'challenger_auc':    round(challenger_auc, 4),
            'champion_auc':      round(champion_auc, 4),
            'improvement':       round(improvement, 4),
            'min_required':      self.min_improvement,
            'decision':          'PROMOTE' if should_promote else 'KEEP_CHAMPION',
            'n_holdout_samples': len(y_holdout),
        }
        self.history.append(result)

        if should_promote:
            print(f'PROMOTING challenger {challenger_version}!')
            print(f'  AUC: {champion_auc:.4f} → {challenger_auc:.4f} (+{improvement:.4f})')
            self.set_champion(challenger, challenger_auc, challenger_version)
        else:
            print(f'Keeping champion {self.champion_ver} (challenger: {improvement:+.4f})')

        return result


# ── Automated retraining pipeline ─────────────────────────────────
class AutoRetrainingPipeline:
    '''
    Automatically retrains model when trigger conditions are met.
    Uses champion-challenger evaluation before promoting.
    '''

    def __init__(self, cc_framework: ChampionChallengerFramework,
                  drift_threshold: float = 0.25,
                  min_auc_threshold: float = 0.90):
        self.cc           = cc_framework
        self.drift_thresh = drift_threshold
        self.min_auc      = min_auc_threshold

    def check_triggers(self, current_auc: float,
                         max_psi: float,
                         days_since_training: int) -> Dict:
        triggers = {
            'performance_degradation': current_auc < self.min_auc,
            'data_drift':              max_psi > self.drift_thresh,
            'scheduled':               days_since_training >= 30,
        }
        should_retrain = any(triggers.values())
        active = [k for k,v in triggers.items() if v]
        return {'should_retrain': should_retrain, 'active_triggers': active, **triggers}

    def retrain(self, X_train: np.ndarray, y_train: np.ndarray,
                 X_holdout: np.ndarray, y_holdout: np.ndarray,
                 version: str) -> Dict:
        print(f'\nRetraining model version {version}...')

        new_model = Pipeline([
            ('sc', StandardScaler()),
            ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42))
        ])
        new_model.fit(X_train, y_train)

        result = self.cc.evaluate_challenger(new_model, X_holdout, y_holdout, version)
        return result


# ── Demonstrate champion-challenger ──────────────────────────────
print('=== Champion-Challenger Retraining Pipeline ===')

data = load_breast_cancer()
X, y = data.data, data.target

# Initial training
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.3, random_state=42)
X_holdout, X_val2, y_holdout, y_val2 = train_test_split(X_val, y_val, test_size=0.5, random_state=42)

champion_v1 = Pipeline([('sc', StandardScaler()),
                          ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42))])
champion_v1.fit(X_tr, y_tr)
v1_auc = roc_auc_score(y_holdout, champion_v1.predict_proba(X_holdout)[:,1])

cc = ChampionChallengerFramework(min_auc_improvement=0.002)
cc.set_champion(champion_v1, v1_auc, 'v1.0')

pipeline = AutoRetrainingPipeline(cc, drift_threshold=0.25, min_auc_threshold=0.92)

# Simulate 4 retraining events
scenarios = [
    ('v2.0', 0.06, 0.92, 15, 'Slight drift'),
    ('v3.0', 0.28, 0.89, 30, 'Major drift + degradation'),
    ('v4.0', 0.12, 0.93, 45, 'Minor drift, scheduled'),
    ('v5.0', 0.05, 0.95, 60, 'Stable, scheduled'),
]

print(f'\n{"Version":>8} {"Triggers":>35} {"Decision":>12}')
print('='*60)
for version, max_psi, cur_auc, days, desc in scenarios:
    triggers = pipeline.check_triggers(cur_auc, max_psi, days)
    active   = ', '.join(triggers['active_triggers']) if triggers['active_triggers'] else 'none'
    if triggers['should_retrain']:
        # Add some noise to X_tr for realistic retraining data
        X_tr_new = X_tr + np.random.normal(0, 0.1, X_tr.shape)
        result   = pipeline.retrain(X_tr_new, y_tr, X_holdout, y_holdout, version)
        decision = result['decision']
    else:
        decision = 'NO_RETRAIN'
    print(f'{version:>8} {active:>35} {decision:>12}  ({desc})')

# Summary
print(f'\nFinal champion: {cc.champion_ver}  AUC={cc.champion_auc:.4f}')
print(f'\nHistory:')
for r in cc.history:
    print(f"  {r['challenger_version']:>6}: challenger={r['challenger_auc']:.4f}  champion={r['champion_auc']:.4f}
Δ={r['improvement']:+.4f}  → {r['decision']}")
```